# Modelling Market Excitement with Hawkes Processes

## Financial Markets Are Not Random In Time

Basic financial models assume events happen independently in time. This outlook represents a poission process, where events occur at a constant rate. 
$$ \lambda(t) = \mu$$
where:
- $\lambda$ is the intensity function
- $\mu$ is the event rate
- $t$ is the time variable, the moment at which we evaluate the intensity

Modelling the market like such assumes that trade arrivals and price movements are random, and order flow is independent. Essentially, this says that an event is independent of past events. 
$$ A_t \bot \{A_{t-1}, A_{t-2}, ...\}$$

However, this is a false assumption as the market is reactive; events effect future events, moves trigger moves, excitement triggers excitement. This can be seen in trading and volatility occuring in clusters, or how large price swings trigger more trading. This phenomena happens for a number of reasons, such as algorithms reacting to order flow or momentum traders reacting to price moves.


## Self-Exciting Processes

A **self-exciting process** describes the notion that events increase the probability of future events. However, the effect fades over time. After each event:
- the event rate increases
- then slowly decays back to normal

## Hawkes Process Intuition

A **Hawkes Process** models the behaviour described above. Each event temporarily increases the event rate. 
$$ \lambda(t) = \mu + \sum_{t_i < t}\alpha e^{-\beta(t-t_i)} $$
where:
- $\mu$ is the baseline trading activity
- $\alpha$ is the strength of excitation
- $\beta$ is the speed of decay of excitation

This means:
- every trade excites the market
- excitement decays exponentially

Hawkes helps explain phenomena such as trade and volatility clustering, flash crashes, and high frequency trading


## Mathematics Background

### Point Processes
**Point processes** model events occuring in constant time. Events are represented as timestamps, $t_1, t_2, ..., t_n$

### Counting Process
**Counting process** : $N(t)$ represents the number of events occured up to time $t$

### Event Intensity
**Event Intensity** : $\lambda(t)$ represents events per unit time. This is the *instantaneous event rate*. For a small time interval dt: $$P(\text{event in }dt) \approx \lambda(t)dt$$
For example: $$\lambda(t) = 5, dt = 0.01 \\ P(\text{event in next 0.01}) = 5 * 0.01 = 0.05 = 5\%$$

### Hawkes Process Definition & Intensity Evolution
- Each event contributes to a temporary increase in intensity
- After multiple events, intensity = sum of decaying events
- $\lambda(t) = \mu + \alpha e^{-\beta(t-t_1)} + \alpha e^{-\beta(t-t_2)} + \alpha e^{-\beta(t-t_3)} + ...$ shows that intensity builds during clusters of events 

### Branching Structure
#### Hawkes Theory Insights
- Events can be interpreted as **offspring** of previous events
    - **Immigrant** : baseline event from $\mu$
    - **Offspring** : triggered by previous event
- Offspring create a branching tree

### Branching Ratio
- **Branching Ratio** measures the expected number of offspring events
$$n = \frac{\alpha}{\beta}$$
- $n < 1$ indicates a *stable system*
- $n \approx 1$ indicates a *highly reflexive market*
- $n > 1$ indicates an *explosive cascade*
    - 0.7 - 09 is the common branching ratio in markets. This means that 70-90% of price moves are caused internally and not in response to new information

In [ ]:
pip install numpy plotly scipy nbformat

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Simulation Parameters ---
n_paths = 3
n_steps = 100
dt = 1

# Hawkes Process Parameters
mu_base = 0.2  # Baseline intensity
alpha = 0.4    # Jump size (self-excitation)
beta = 0.8     # Decay rate

np.random.seed(42)

# --- Hawkes Process Simulation ---
N = np.zeros((n_paths, n_steps + 1))
lambdas = np.zeros((n_paths, n_steps + 1))
lambdas[:, 0] = mu_base

for t in range(1, n_steps + 1):
    increments = np.random.poisson(lambdas[:, t-1] * dt)
    N[:, t] = N[:, t - 1] + increments
    lambdas[:, t] = mu_base + (lambdas[:, t-1] - mu_base) * np.exp(-beta * dt) + alpha * increments

time_grid = np.arange(n_steps + 1)

# Global Y ranges
max_y_N = int(np.max(N)) + 5
max_y_L = np.max(lambdas) * 1.1

# --- Styling Parameters ---
neon_colors = ['#ff00ff', '#39ff14', '#00ffff']  # Pink, Green, Cyan
path_width = 1.8
opacity_val = .9

# --- Animation Frames ---
frames = []
for t in range(n_steps + 1):
    time_show = time_grid[:t + 1]
    
    # Top Row: N(t) Traces
    traces_N = [
        go.Scatter(
            x=time_show, y=N[i, :t+1], 
            mode='lines', line=dict(color=neon_colors[i], width=path_width, shape='hv'),
            opacity=opacity_val,
            name=f"Path {i+1}", legendgroup=f"group{i}"
        ) for i in range(n_paths)
    ]
    vline_N = go.Scatter(x=[t, t], y=[-1, max_y_N], mode='lines', line=dict(color='yellow', dash='dash'), showlegend=False)
    
    # Bottom Row: Lambda(t) Traces
    traces_L = [
        go.Scatter(
            x=time_show, y=lambdas[i, :t+1], 
            mode='lines', line=dict(color=neon_colors[i], width=path_width), 
            opacity=opacity_val,
            showlegend=False, legendgroup=f"group{i}"
        ) for i in range(n_paths)
    ]
    vline_L = go.Scatter(x=[t, t], y=[0, max_y_L], mode='lines', line=dict(color='yellow', dash='dash'), showlegend=False)

    # Frame creation
    frames.append(go.Frame(
        data=[*traces_N, vline_N, *traces_L, vline_L],
        name=f"step{t}"
    ))

# --- Figure Structure ---
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=[
        "Cumulative Events N(t)", 
        "Underlying Self-Exciting Rate λ(t)"
    ]
)

# Populate initial traces
init_data = frames[0].data

# Top Row Traces (3 paths + 1 vline)
for i in range(n_paths): fig.add_trace(init_data[i], row=1, col=1)
fig.add_trace(init_data[n_paths], row=1, col=1)

# Bottom Row Traces (3 paths + 1 vline)
offset = n_paths + 1
for i in range(n_paths): fig.add_trace(init_data[offset + i], row=2, col=1)
fig.add_trace(init_data[offset + n_paths], row=2, col=1)

fig.frames = frames

# --- Static Marker for Baseline mu (Applied to Bottom Row) ---
fig.add_hline(
    y=mu_base, row=2, col=1,
    line_width=1.5, line_dash="dash", line_color="rgba(255, 255, 255, 0.6)", 
    annotation_text="μ (Base Rate)", annotation_position="top left", annotation_font_color="white"
)

# --- Slider Configuration ---
sliders = [dict(
    active=0, currentvalue={"prefix": "Time Step: "}, pad={"t": 0}, x=0.25, len=0.75, y=-0.05,
    steps=[dict(method="animate", args=[[f"step{k}"], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))], label=str(k)) for k in range(n_steps + 1)]
)]

# --- Layout ---
fig.update_layout(
    height=800, width=1000,
    title_text=f"Hawkes Process Dynamics (μ={mu_base}, α={alpha}, β={beta})",
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(color="black"),
    showlegend=True, legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    sliders=sliders, margin=dict(b=80, t=100),
    updatemenus=[{
        'type': 'buttons', 'x': 0.0, 'y': -0.1, 'xanchor': 'left', 'yanchor': 'top', 'direction': 'left', 'showactive': False,
        'buttons': [
            {'label': '▶ Play', 'method': 'animate', 'args': [None, {'frame': {'duration': 40, 'redraw': True}, 'fromcurrent': True}]},
            {'label': '⏸ Pause', 'method': 'animate', 'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]}
        ]
    }]
)

# Axes formatting
for r in [1, 2]:
    fig.update_xaxes(showgrid=True, gridcolor='rgba(128,128,128,0.3)', row=r, col=1)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.3)', row=r, col=1)

fig.update_yaxes(title_text='Events N(t)', range=[-1, max_y_N], row=1, col=1)
fig.update_xaxes(title_text='Time (t)', range=[0, n_steps], row=2, col=1)
fig.update_yaxes(title_text='Rate λ(t)', range=[0, max_y_L], row=2, col=1)

fig.show()